<a href="https://cognitiveclass.ai"><img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DL0101EN-SkillsNetwork/images/IDSN-logo.png" width="400"> </a>

# Keras ile Regresyon Modelleri

Tahmini süre: **45** dakika

## Introduction


Videolarda da tartıştığımız gibi, PyTorch ve TensorFlow gibi daha güçlü kütüphanelerin popülaritesine rağmen, kullanımları kolay değil ve öğrenme eğrileri dik. Bu nedenle, derin öğrenmeye yeni başlayanlar için Keras kütüphanesinden daha iyi bir kütüphane yok.

Keras, derin öğrenme modelleri oluşturmak için yüksek seviyeli bir API'dir. Kullanım kolaylığı ve sözdizimsel basitliği sayesinde hızlı geliştirmeyi kolaylaştırdığı için beğeni kazanmıştır. Bu laboratuvarda ve bu kursun diğer laboratuvarlarında göreceğiniz gibi, Keras ile sadece birkaç satır kodla çok karmaşık bir derin öğrenme ağı oluşturmak mümkün. Diğer kurslarda PyTorch ve TensorFlow kullanarak derin modeller oluşturmayı öğrendikten sonra Keras'ı daha da çok takdir edeceksiniz.

Bu nedenle, bu laboratuvarda, Keras kütüphanesini kullanarak bir regresyon modeli oluşturmayı öğreneceksiniz.

## Bu Notebook'un Amaçları
* Keras kütüphanesini kullanarak bir regresyon modeli oluşturma
* Veri setini indirme ve temizleme
* Bir sinir ağı oluşturma
* Ağı eğitme ve test etme

<h2>İçindekiler</h2>

<div class="alert alert-block alert-info" style="margin-top: 20px">

<font size = 4>
1. <a href="#Veri Kümesini İndir ve Temizle">Veri Kümesini İndir ve Temizle</a><br>
2. <a href="#Keras Paketlerini İçe Aktar">Keras Paketlerini İçe Aktar</a><br>
3. <a href="#Sinir Ağı Oluştur">Sinir Ağı Oluştur</a><br>
4. <a href="#Ağı Eğit ve Test Et">Ağı Eğit ve Test Et</a><br>

</font>
</div>

Let's start by importing the <em>pandas</em> and the Numpy libraries.


In [3]:
# All Libraries required for this lab are listed below. 

!pip install numpy==2.0.2
!pip install pandas==2.2.2
!pip install tensorflow_cpu==2.18.0


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 126.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 78.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.3/230.3 MB 42.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 48.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 54.3 MB/s eta 0:00:00


#### Keras'ı kullanmak için, TensorFlow gibi bir arka uç çerçevesi de kurmanız gerekecektir.

TensorFlow 2.16 veya üstünü kurarsanız, Keras varsayılan olarak kurulacaktır.

Daha küçük veri kümeleriyle çalıştığımız için TensorFlow'un CPU sürümünü kullanıyoruz.

Daha büyük veri kümelerinin işlenmesini hızlandırmak için makinenize TensorFlow'un GPU sürümünü kurabilirsiniz.

#### TensorFlow uyarı mesajlarını bastırma
TensorFlow için CPU mimarisi kullanımından kaynaklanan uyarı mesajlarını bastırmak için aşağıdaki kodu kullanıyoruz.

GPU mimarisi kullanıyorsanız bu satırları **yorum satırı haline getirmek** isteyebilirsiniz.

In [4]:
import os
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

In [5]:
import pandas as pd
import numpy as np
import keras

import warnings
warnings.simplefilter('ignore', FutureWarning)

Videolarda kullandığımız aynı veri setini kullanacağız.

<strong>Veri seti, farklı beton örneklerinin, yapımında kullanılan farklı bileşenlerin hacimlerine bağlı olarak basınç dayanımıyla ilgilidir. Bileşenler şunlardır:</strong>

* Çimento
* Yüksek fırın cürufu
* Uçucu kül
* Su
* Süper akışkanlaştırıcı
* İri agrega
* İnce agrega

## Download and Clean the Data Set


Let's download the data and read it into a <em>pandas</em> dataframe.


In [6]:
filepath='https://s3-api.us-geo.objectstorage.softlayer.net/cf-courses-data/CognitiveClass/DL0101EN/labs/data/concrete_data.csv'
concrete_data = pd.read_csv(filepath)

concrete_data.head()

,Cement,Blast Furnace Slag,Fly Ash,Water,Superplasticizer,Coarse Aggregate,Fine Aggregate,Age,Strength
0,540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28,79.99
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28,61.89
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270,40.27
3,332.5,142.5,0.0,228.0,0.0,932.0,594.0,365,41.05
4,198.6,132.4,0.0,192.0,0.0,978.4,825.5,360,44.30


İlk beton numunesinde 540 metreküp çimento, 0 metreküp yüksek fırın cürufu, 0 metreküp uçucu kül, 162 metreküp su, 2,5 metreküp süperplastikleştirici, 1040 metreküp iri agrega ve 676 metreküp ince agrega bulunmaktadır. 28 günlük bu beton karışımının basınç dayanımı 79,99 MPa'dır.

#### Let's check how many data points we have


In [7]:
concrete_data.shape

(1030, 9)

Dolayısıyla, modelimizi eğitmek için yaklaşık 1000 örnek bulunmaktadır. Örnek sayısının az olması nedeniyle, eğitim verilerine aşırı uyum sağlamamaya dikkat etmeliyiz.

Let's check the dataset for any missing values.


In [8]:
concrete_data.describe()

,Cement,Blast Furnace Slag,Fly Ash,Water,Superplasticizer,Coarse Aggregate,Fine Aggregate,Age,Strength
count,1030.000000,1030.000000,1030.000000,1030.000000,1030.000000,1030.000000,1030.000000,1030.000000,1030.000000
mean,281.167864,73.895825,54.188350,181.567282,6.204660,972.918932,773.580485,45.662136,35.817961
std,104.506364,86.279342,63.997004,21.354219,5.973841,77.753954,80.175980,63.169912,16.705742
min,102.000000,0.000000,0.000000,121.800000,0.000000,801.000000,594.000000,1.000000,2.330000
25%,192.375000,0.000000,0.000000,164.900000,0.000000,932.000000,730.950000,7.000000,23.710000
50%,272.900000,22.000000,0.000000,185.000000,6.400000,968.000000,779.500000,28.000000,34.445000
75%,350.000000,142.950000,118.300000,192.000000,10.200000,1029.400000,824.000000,56.000000,46.135000
max,540.000000,359.400000,200.100000,247.000000,32.200000,1145.000000,992.600000,365.000000,82.600000


In [9]:
concrete_data.isnull().sum()

Cement                0
Blast Furnace Slag    0
Fly Ash               0
Water                 0
Superplasticizer      0
Coarse Aggregate      0
Fine Aggregate        0
Age                   0
Strength              0
dtype: int64

The data looks very clean and is ready to be used to build our model.


#### Split data into predictors and target


Bu problemdeki hedef değişken beton numunesinin dayanımıdır. Dolayısıyla, tahmin edicilerimiz diğer tüm sütunlar olacaktır.

In [10]:
concrete_data_columns = concrete_data.columns

In [12]:
predictors = concrete_data[concrete_data_columns[concrete_data_columns != 'Strength']] # Güç hariç tüm sütunlar
target = concrete_data['Strength'] # Strength column

<a id="item2"></a>


Let's do a quick sanity check of the predictors and the target dataframes.


In [13]:
predictors.head()

,Cement,Blast Furnace Slag,Fly Ash,Water,Superplasticizer,Coarse Aggregate,Fine Aggregate,Age
0,540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270
3,332.5,142.5,0.0,228.0,0.0,932.0,594.0,365
4,198.6,132.4,0.0,192.0,0.0,978.4,825.5,360


In [14]:
target.head()

0    79.99
1    61.89
2    40.27
3    41.05
4    44.30
Name: Strength, dtype: float64

Son olarak, son adım, ortalamayı çıkarıp standart sapmaya bölerek verileri normalleştirmektir.

In [15]:
predictors_norm = (predictors - predictors.mean()) / predictors.std()
predictors_norm.head()

,Cement,Blast Furnace Slag,Fly Ash,Water,Superplasticizer,Coarse Aggregate,Fine Aggregate,Age
0,2.476712,-0.856472,-0.846733,-0.916319,-0.620147,0.862735,-1.217079,-0.279597
1,2.476712,-0.856472,-0.846733,-0.916319,-0.620147,1.055651,-1.217079,-0.279597
2,0.491187,0.795140,-0.846733,2.174405,-1.038638,-0.526262,-2.239829,3.551340
3,0.491187,0.795140,-0.846733,2.174405,-1.038638,-0.526262,-2.239829,5.055221
4,-0.790075,0.678079,-0.846733,0.488555,-1.038638,0.070492,0.647569,4.976069


Ağımızı oluştururken bu sayıya ihtiyacımız olacağı için, tahmin edici sayısını *n_cols* değişkenine kaydedelim.

In [17]:
n_cols = predictors_norm.shape[1] # number of predictors
print(n_cols)

8


<a id="item1"></a>


##  Import Keras Packages

##### Let's import the rest of the packages from the Keras library that we will need to build our regression model.


In [18]:
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import Input

## Build a Neural Network


Let's define a function that defines our regression model for us so that we can conveniently call it to create our model.


In [19]:
# define regression model
def regression_model():                          # regresyon problemi için bir sinir ağı modeli oluşturan fonksiyon   
    model = Sequential()                         # katmanların sırayla eklendiği bir neural network modeli oluşturur   
    model.add(Input(shape=(n_cols,)))            # giriş katmanı: veri setindeki özellik (feature) sayısı kadar input alır    
    model.add(Dense(50, activation='relu'))      # 1. gizli katman: 50 nöron içerir, ReLU aktivasyonu kullanır      
    model.add(Dense(1))                          # çıkış katmanı: tek bir sayısal değer üretir (regression çıktısı)    
    model.compile(optimizer='adam',              # optimizer: Adam algoritması ile ağırlıkları günceller
                  loss='mean_squared_error')     # loss fonksiyonu: tahmin ile gerçek değer arasındaki hatayı MSE ile ölçer   
    return model                                 # oluşturulan modeli fonksiyonun çıktısı olarak döndürür

Yukarıdaki fonksiyon, her biri 50 gizli birimden oluşan iki gizli katmana sahip bir model oluşturur.

## Ağı Eğitin ve Test Edin

Şimdi modelimizi oluşturmak için fonksiyonu çağıralım.

In [20]:
# build the model
model = regression_model()

Ardından, *fit* yöntemini kullanarak modeli aynı anda eğitecek ve test edeceğiz. Verilerin %30'unu doğrulama için ayıracağız ve modeli 100 epoch boyunca eğiteceğiz.

In [21]:
# fit the model
model.fit(predictors_norm, target, validation_split=0.3, epochs=100, verbose=2)

Epoch 1/100
23/23 - 1s - 39ms/step - loss: 1716.1414 - val_loss: 1232.3777
Epoch 2/100
23/23 - 0s - 7ms/step - loss: 1676.4047 - val_loss: 1203.9286
Epoch 3/100
23/23 - 0s - 7ms/step - loss: 1635.6422 - val_loss: 1174.5001
Epoch 4/100
23/23 - 0s - 7ms/step - loss: 1593.1467 - val_loss: 1143.5619
Epoch 5/100
23/23 - 0s - 7ms/step - loss: 1546.0172 - val_loss: 1110.6031
Epoch 6/100
23/23 - 0s - 6ms/step - loss: 1494.5767 - val_loss: 1075.6281
Epoch 7/100
23/23 - 0s - 6ms/step - loss: 1437.6556 - val_loss: 1037.7419
Epoch 8/100
23/23 - 0s - 6ms/step - loss: 1375.4891 - val_loss: 995.6945
Epoch 9/100
23/23 - 0s - 6ms/step - loss: 1306.4540 - val_loss: 951.4463
Epoch 10/100
23/23 - 0s - 6ms/step - loss: 1233.3306 - val_loss: 902.7948
Epoch 11/100
23/23 - 0s - 6ms/step - loss: 1153.9609 - val_loss: 853.1514
Epoch 12/100
23/23 - 0s - 6ms/step - loss: 1072.0830 - val_loss: 800.3591
Epoch 13/100
23/23 - 0s - 6ms/step - loss: 987.4769 - val_loss: 747.5723
Epoch 14/100
23/23 - 0s - 6ms/step - los

<strong>Tahmin veya değerlendirme için kullanabileceğiniz diğer işlevler hakkında bilgi edinmek için bu [bağlantıya](https://keras.io/models/sequential/) bakabilirsiniz.</strong>

Aşağıdakileri değiştirmekte ve her değişikliğin modelin performansına etkisini not etmekte özgürsünüz:

1. Gizli katmanlardaki nöron sayısını artırın veya azaltın
2. Daha fazla gizli katman ekleyin
3. Epoch sayısını artırın

<h3>Practice Exercise 1</h3>


Şimdi aynı veri setini kullanarak, her biri 50 düğümlü ve ReLU aktivasyon fonksiyonlu beş gizli katmana, tek bir çıkış katmanına sahip ve Adam optimizasyon algoritması kullanılarak optimize edilmiş bir regresyon modeli oluşturmayı deneyin.

In [ ]:
def regression_model():
    input_colm = predictors_norm.shape[1] 
    # create model
    model = Sequential()
    model.add(Input(shape=(input_colm,)))  
    model.add(Dense(50, activation='relu'))  
    model.add(Dense(50, activation='relu'))
    model.add(Dense(50, activation='relu')) 
    model.add(Dense(50, activation='relu'))
    model.add(Dense(50, activation='relu'))  
    model.add(Dense(1))  # Output layer
    
    # compile model
    model.compile(optimizer='adam', loss='mean_squared_error')
    return model



Double-click <b>here</b> for the solution.

<!-- Your answer is below:
def regression_model():
    input_colm = predictors_norm.shape[1] # Number of input features
    # create model
    model = Sequential()
    model.add(Input(shape=(input_colm,)))  # Set the number of input features 
    model.add(Dense(50, activation='relu'))  
    model.add(Dense(50, activation='relu'))
    model.add(Dense(50, activation='relu')) 
    model.add(Dense(50, activation='relu'))
    model.add(Dense(50, activation='relu'))  
    model.add(Dense(1))  # Output layer
    
    # compile model
    model.compile(optimizer='adam', loss='mean_squared_error')
    return model

-->


<h3>Practice Exercise 2</h3>


Verilerin %10'unu doğrulama için ayırarak ve modeli 100 epoch boyunca eğiterek, fit() yöntemini kullanarak modeli eş zamanlı olarak eğitin ve değerlendirin.

In [22]:
model.fit(predictors_norm, target, validation_split=0.3, epochs=100, verbose=2)

Epoch 1/100
23/23 - 0s - 8ms/step - loss: 92.1692 - val_loss: 104.8121
Epoch 2/100
23/23 - 0s - 6ms/step - loss: 90.7458 - val_loss: 104.4025
Epoch 3/100
23/23 - 0s - 6ms/step - loss: 89.4379 - val_loss: 102.9142
Epoch 4/100
23/23 - 0s - 6ms/step - loss: 88.1720 - val_loss: 101.9091
Epoch 5/100
23/23 - 0s - 6ms/step - loss: 87.0242 - val_loss: 101.3236
Epoch 6/100
23/23 - 0s - 6ms/step - loss: 85.5682 - val_loss: 101.5918
Epoch 7/100
23/23 - 0s - 6ms/step - loss: 84.2233 - val_loss: 100.3918
Epoch 8/100
23/23 - 0s - 6ms/step - loss: 82.9011 - val_loss: 101.4206
Epoch 9/100
23/23 - 0s - 7ms/step - loss: 81.7394 - val_loss: 99.8848
Epoch 10/100
23/23 - 0s - 6ms/step - loss: 80.6396 - val_loss: 98.5441
Epoch 11/100
23/23 - 0s - 6ms/step - loss: 79.3283 - val_loss: 99.7539
Epoch 12/100
23/23 - 0s - 6ms/step - loss: 78.2448 - val_loss: 98.8277
Epoch 13/100
23/23 - 0s - 6ms/step - loss: 77.2400 - val_loss: 98.6174
Epoch 14/100
23/23 - 0s - 6ms/step - loss: 76.0575 - val_loss: 97.9285
Epoch 1

Double-click <b>here</b> for the solution.

<!-- Your answer is below:
# build the model
model = regression_model()
model.fit(predictors_norm, target, validation_split=0.1, epochs=100, verbose=2)

-->


Sonuçlara dayanarak şunları fark ediyoruz:

- Modele daha fazla gizli katman eklemek, modelin veriler içindeki karmaşık ilişkileri öğrenme ve temsil etme kapasitesini artırır. Bu, modelin daha iyi tanımlama yapmasını sağlar ve sonuç olarak model, eğitim verilerine daha etkili bir şekilde uyum sağlar ve tahminlerini potansiyel olarak iyileştirir.

- Doğrulama için ayrılan veri kümesinin oranını azaltarak ve daha büyük bir bölümünü eğitim için kullanarak, modelin öğrenmek için daha fazla örneğe erişimi olur. Bu ek eğitim verisi, modelin altta yatan eğilimleri daha iyi anlamasına yardımcı olur ve bu da genel performansı iyileştirebilir.

### Thank you for completing this lab!

This notebook was created by [Alex Aklson](https://www.linkedin.com/in/aklson/). I hope you found this lab interesting and educational. Feel free to contact me if you have any questions!


<!--
## Change Log

|  Date (YYYY-MM-DD) |  Version | Changed By  |  Change Description |
|---|---|---|---|
| 2024-11-20  | 3.0  | Aman  |  Updated the library versions to current |
| 2020-09-21  | 2.0  | Srishti  |  Migrated Lab to Markdown and added to course repo in GitLab |



<hr>

## <h3 align="center"> © IBM Corporation. All rights reserved. <h3/>


## <h3 align="center"> &#169; IBM Corporation. All rights reserved. <h3/>

